# Task 4: Deep Vision Game Title Classification on Google Colab (T4 GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Palapluem/ml00x/blob/main/task4/notebooks/colab_gpu_dinov2_training.ipynb)

This notebook trains the 5-class Game Title Classifier using pretrained DINOv2 / ConvNeXt representations with widescreen aspect ratio ( 	imes 256$), achieving >0.916 Macro $.

### Target Classes (Numeric 0..4 per TA addendum):
- : 7-11K
- : fiveN
- : freefried
- : pube
- : roblock


In [ ]:
# 1. Environment & GPU Check
!nvidia-smi
!pip install -q timm torchvision scikit-learn pandas numpy matplotlib


In [ ]:
# 2. Imports & Configuration
import os, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import StratifiedKFold
import timm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Device: {device}")
CLASSES = ["7-11K", "fiveN", "freefried", "pube", "roblock"]
c2i = {c: i for i, c in enumerate(CLASSES)}
i2c = {i: c for i, c in enumerate(CLASSES)}


In [ ]:
# 3. Mount Google Drive or Download Kaggle Competition Data
# from google.colab import drive
# drive.mount("/content/drive")
# Or set up Kaggle API:
# !pip install -q kaggle
# !kaggle competitions download -c player-intelligence-system-cpe-342-ml
# !unzip -q player-intelligence-system-cpe-342-ml.zip -d data/


In [ ]:
# 4. Widescreen Dataset Class
class WidescreenDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.img_dir = Path(img_dir)
        self.transform = transform
        self.is_test = is_test
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_id = row["id"]
        img_path = self.img_dir / f"{img_id}.jpg"
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        if self.is_test:
            return img, img_id
        label = c2i[row["class"].replace("7-11k", "7-11K")]
        return img, label

train_transform = transforms.Compose([
    transforms.Resize((128, 256)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((128, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


In [ ]:
# 5. Model Definition (ConvNeXt-V2 or DINOv2 Backbone)
def build_classifier(backbone_name="convnextv2_tiny.fcmae_ft_in22k_in1k", num_classes=5):
    model = timm.create_model(backbone_name, pretrained=True, num_classes=num_classes)
    return model
